# Free catalog builder (no API key, no cost)
Works in **Google Colab** or on **your own PC** (Jupyter / VS Code). Run the cells top to bottom; do the 10-post test (step 4) first.

> `gm` = typical weight of the category and `price` is formula-based, because a photo can't show weight. Edit those numbers in step 2.

## 1. Install (installs into the same Python this notebook uses)
The first time takes a few minutes (PyTorch is large). **Restart the kernel after it finishes** if it asks.

In [11]:
%pip install -q torch transformers pillow pandas numpy requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Write the script
Edit **PRICING** and **GM_TYPICAL** in this cell, then run it.

In [12]:
%%writefile free_catalog.py
#!/usr/bin/env python3
"""
free_catalog.py  -  100% FREE version (no API key, no paid service)
====================================================================
Uses the open-source CLIP model (Hugging Face) to look at every image and predict:
    name, desc, category, image, image_urls, sizes, colors, gm, gender, price

How each field is produced
--------------------------
category   CLIP zero-shot (ring / bracelet / necklace / ...), Arabic caption keywords override
colors     CLIP zero-shot on the colour of the item (metal tone / stone colour)
gender     CLIP zero-shot + caption keywords (defaults to Women for jewelry)
name/desc  built from template + predicted colour / category / motif (heart, flower, pearl ...)
sizes      sensible default per category
gm         AVERAGE weight of the category (a photo can't show weight) - edit GM_TYPICAL
price      gm * silver price * margin + making fee - edit the PRICING block
Duplicates
  same image                    -> pHash close            -> add URL to image_urls
  same design, different colour -> CLIP similarity on
                                   GREYSCALE images       -> add URL to image_urls + new colour to colors

Run:   python free_catalog.py --csv telegram_images.csv --out catalog.csv
"""

import argparse
import hashlib
import json
import sys
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from PIL import Image, ImageOps

# --------------------------------------------------------------------------- #
# CONFIG
# --------------------------------------------------------------------------- #
CATEGORIES = ["Rings", "Bracelets", "Necklaces", "Earrings", "Accessories", "Sets", "Bags"]

CLIP_MODEL = "openai/clip-vit-base-patch32"   # fast. Better but slower: "openai/clip-vit-large-patch14"

DUP_THRESHOLD = 5        # pHash Hamming distance  -> same image
DESIGN_SIM = 0.93        # cosine similarity of greyscale CLIP embeddings -> same design (other colour)
                         # raise (0.95) if different items get merged, lower (0.90) if variants are missed
MERGE_WHOLE_POST = False # True = every image of a Telegram post becomes ONE product

SECOND_COLOR_MIN = 0.30  # a 2nd colour is added only if its probability is >= this

# ---- PRICING (PLACEHOLDERS - replace with your real numbers) ----
CURRENCY = "EGP"
SILVER_PRICE_PER_GRAM = 70.0
MARGIN = 0.60
MAKING_FEE = {"Rings": 120, "Bracelets": 180, "Necklaces": 200, "Earrings": 120,
              "Accessories": 100, "Sets": 400, "Bags": 0}
BAG_PRICE = {"Small": 650, "Medium": 900, "Large": 1200}
ROUND_TO = 10

# A photo cannot show weight, so gm = typical weight of the category. Edit to match your stock.
GM_TYPICAL = {"Rings": 5.0, "Bracelets": 15.0, "Necklaces": 12.0, "Earrings": 5.0,
              "Accessories": 8.0, "Sets": 35.0, "Bags": None}

DEFAULT_SIZES = {"Rings": "Adjustable", "Bracelets": "Adjustable", "Necklaces": "Adjustable (40-45 cm)",
                 "Earrings": "One size", "Accessories": "One size", "Sets": "One size", "Bags": "Medium"}

SINGULAR = {"Rings": "Ring", "Bracelets": "Bracelet", "Necklaces": "Necklace", "Earrings": "Earrings",
            "Accessories": "Accessory", "Sets": "Jewelry Set", "Bags": "Bag"}
CLIP_NOUN = {"Rings": "ring", "Bracelets": "bracelet", "Necklaces": "necklace", "Earrings": "pair of earrings",
             "Accessories": "jewelry accessory", "Sets": "jewelry set", "Bags": "handbag"}

COLORS = ["Silver", "Gold", "Rose Gold", "Black", "White", "Red", "Pink", "Blue", "Green",
          "Purple", "Turquoise", "Brown", "Multicolor"]

CATEGORY_PROMPTS = {
    "Rings": ["a photo of a ring", "a photo of a finger ring jewelry"],
    "Bracelets": ["a photo of a bracelet", "a photo of a bangle jewelry"],
    "Necklaces": ["a photo of a necklace", "a photo of a pendant necklace on a chain"],
    "Earrings": ["a photo of earrings", "a photo of a pair of earrings"],
    "Accessories": ["a photo of a hair accessory", "a photo of an anklet, brooch, watch or keychain"],
    "Sets": ["a photo of a jewelry set with a necklace and matching earrings", "a photo of a matching jewelry set"],
    "Bags": ["a photo of a handbag", "a photo of a purse"],
}

GENDER_PROMPTS = {
    "Women": ["jewelry for women", "a feminine delicate accessory"],
    "Men": ["jewelry for men", "a masculine bold accessory"],
    "Kids": ["jewelry for kids", "a cute accessory for a child"],
}

MOTIFS = ["Heart", "Flower", "Butterfly", "Star", "Cross", "Pearl", "Zircon", "Evil Eye", "Initial Letter",
          "Moon", "Clover", "Bow", "Snake", "Infinity", "Minimalist", "Twisted", "Chain Link"]
BAG_MOTIFS = ["Leather", "Chain Strap", "Mini", "Tote", "Crossbody", "Clutch", "Backpack"]
MOTIF_MIN = 0.25

# Arabic / English caption keywords (substring match). Captions in your file are mostly empty,
# but when present they are more reliable than the image model.
CAPTION_CATEGORY = {
    "خاتم": "Rings", "خواتم": "Rings", "ring": "Rings",
    "اسورة": "Bracelets", "أسورة": "Bracelets", "إسورة": "Bracelets", "سوار": "Bracelets",
    "غويشة": "Bracelets", "bracelet": "Bracelets",
    "كوليه": "Necklaces", "سلسلة": "Necklaces", "عقد": "Necklaces", "دلاية": "Necklaces", "necklace": "Necklaces",
    "حلق": "Earrings", "اقراط": "Earrings", "earring": "Earrings",
    "طقم": "Sets", "set": "Sets",
    "شنطة": "Bags", "شنط": "Bags", "حقيبة": "Bags", "bag": "Bags",
}
CAPTION_GENDER = {"حريمي": "Women", "نسائي": "Women", "بناتي": "Women", "women": "Women",
                  "رجالي": "Men", "men": "Men",
                  "اطفال": "Kids", "أطفال": "Kids", "kids": "Kids"}
CAPTION_STONE = {"زركون": "Zircon", "لؤلؤ": "Pearl", "ياقوت": "Ruby", "فيروز": "Turquoise", "زمرد": "Emerald"}

IMG_DIR = Path("cache/images")
MAX_SIDE = 512
DOWNLOAD_WORKERS = 16


# --------------------------------------------------------------------------- #
# DOWNLOAD + HASH
# --------------------------------------------------------------------------- #
def cache_path(url):
    return IMG_DIR / (hashlib.sha1(url.encode()).hexdigest() + ".jpg")


def download(url):
    p = cache_path(url)
    if p.exists() and p.stat().st_size > 0:
        return p
    for attempt in range(4):
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            p.write_bytes(r.content)
            return p
        except Exception:
            time.sleep(1.5 * (attempt + 1))
    return None


def load_small(path):
    img = Image.open(path).convert("RGB")
    img.thumbnail((MAX_SIDE, MAX_SIDE))
    return img


def _dct_matrix(n=32):
    k, i = np.arange(n)[:, None], np.arange(n)[None, :]
    m = np.cos(np.pi * (2 * i + 1) * k / (2 * n))
    m[0] *= 1 / np.sqrt(2)
    return m * np.sqrt(2 / n)


_DCT = _dct_matrix(32)
_POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def phash(img):
    g = np.asarray(img.convert("L").resize((32, 32), Image.LANCZOS), dtype=np.float32)
    low = (_DCT @ g @ _DCT.T)[:8, :8].flatten()
    bits = (low > np.median(low[1:])).astype(np.uint8)
    v = 0
    for b in bits:
        v = (v << 1) | int(b)
    return np.uint64(v)


def hamming_matrix(h):
    x = np.bitwise_xor(h[:, None], h[None, :])
    return _POP[x.view(np.uint8).reshape(len(h), len(h), 8)].sum(-1)


# --------------------------------------------------------------------------- #
# CLIP
# --------------------------------------------------------------------------- #
def _l2(a):
    return a / np.linalg.norm(a, axis=-1, keepdims=True)


class Clip:
    def __init__(self, name=CLIP_MODEL):
        try:
            import torch
            from transformers import CLIPModel, CLIPProcessor
        except ImportError as e:
            raise SystemExit(f"Missing package: {e.name}. Run:  pip install torch transformers   "
                             "(then restart the notebook kernel)")
        self.torch = torch
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = CLIPModel.from_pretrained(name).to(self.device).eval()
        self.proc = CLIPProcessor.from_pretrained(name)
        print(f"CLIP loaded on {self.device}")

    @staticmethod
    def _tensor(x):
        return x if hasattr(x, "shape") else x.pooler_output

    def images(self, imgs, bs=32):
        out = []
        with self.torch.no_grad():
            for i in range(0, len(imgs), bs):
                inp = self.proc(images=imgs[i:i + bs], return_tensors="pt").to(self.device)
                out.append(self._tensor(self.model.get_image_features(**inp)).cpu().numpy())
        return _l2(np.concatenate(out))

    def texts(self, texts):
        with self.torch.no_grad():
            inp = self.proc(text=texts, return_tensors="pt", padding=True).to(self.device)
            return _l2(self._tensor(self.model.get_text_features(**inp)).cpu().numpy())


def softmax(x):
    e = np.exp(x - x.max(-1, keepdims=True))
    return e / e.sum(-1, keepdims=True)


class Labeler:
    """Zero-shot label probabilities from image embeddings."""

    def __init__(self, clip):
        self.clip = clip
        self.cat_mat = self._ensemble(CATEGORY_PROMPTS, CATEGORIES)
        self.gender_keys = list(GENDER_PROMPTS)
        self.gender_mat = self._ensemble(GENDER_PROMPTS, self.gender_keys)
        self._color, self._motif = {}, {}

    def _ensemble(self, prompts, keys):
        return _l2(np.stack([self.clip.texts(prompts[k]).mean(0) for k in keys]))

    def _probs(self, emb, mat):
        return softmax(100.0 * emb @ mat.T)

    def category(self, emb):
        return self._probs(emb, self.cat_mat)

    def gender(self, emb):
        return self._probs(emb, self.gender_mat)

    def colors(self, emb, category):
        noun = CLIP_NOUN[category]
        if noun not in self._color:
            self._color[noun] = self.clip.texts([f"a {c.lower()} {noun}" for c in COLORS])
        return self._probs(emb, self._color[noun])

    def motifs(self, emb, category):
        names = BAG_MOTIFS if category == "Bags" else MOTIFS
        noun = CLIP_NOUN[category]
        key = (noun, category == "Bags")
        if key not in self._motif:
            self._motif[key] = self.clip.texts([f"a {noun} with a {m.lower()} design" for m in names])
        return names, self._probs(emb, self._motif[key])


# --------------------------------------------------------------------------- #
# CLUSTERING
# --------------------------------------------------------------------------- #
class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]
            a = self.p[a]
        return a

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[rb] = ra


def cluster_images(hashes, gray_emb, categories, post_ids, merge_post=MERGE_WHOLE_POST):
    n = len(hashes)
    uf = UnionFind(n)
    dist = hamming_matrix(hashes)
    sim = gray_emb @ gray_emb.T
    stats = Counter()
    for i in range(n):
        for j in range(i + 1, n):
            if dist[i, j] <= DUP_THRESHOLD:
                uf.union(i, j)
                stats["same_image"] += 1
            elif sim[i, j] >= DESIGN_SIM and categories[i] == categories[j]:
                uf.union(i, j)
                stats["same_design"] += 1
    if merge_post:
        first = {}
        for i, pid in enumerate(post_ids):
            if pid in first:
                uf.union(first[pid], i)
            else:
                first[pid] = i
    groups = {}
    for i in range(n):
        groups.setdefault(uf.find(i), []).append(i)
    return list(groups.values()), stats


# --------------------------------------------------------------------------- #
# ROW BUILDING
# --------------------------------------------------------------------------- #
def keyword(text, table):
    t = (text or "").lower()
    for k, v in table.items():
        if k in t:
            return v
    return None


def pick_gender(p):
    if p["Kids"] > 0.5:
        return "Kids"
    if p["Men"] > 0.5:
        return "Men"
    if p["Men"] > 0.35:
        return "Unisex"
    return "Women"


def estimate_price(category, gm, size):
    if category == "Bags":
        base = next((v for k, v in BAG_PRICE.items() if k.lower() in size.lower()), BAG_PRICE["Medium"])
    else:
        base = gm * SILVER_PRICE_PER_GRAM * (1 + MARGIN) + MAKING_FEE.get(category, 0)
    return int(round(base / ROUND_TO) * ROUND_TO)


def build_row(idx, urls, post_urls, captions, cat_p, gender_p, color_p, motif_p):
    """idx: image indexes of this product; cat/gender/color_p: per-image arrays; motif_p: per-image {motif: prob}."""
    caption = next((captions[i] for i in idx if captions[i].strip()), "")

    mean_cat = cat_p[idx].mean(0)
    category = keyword(caption, CAPTION_CATEGORY) or CATEGORIES[int(mean_cat.argmax())]
    confidence = float(mean_cat.max())

    gender = keyword(caption, CAPTION_GENDER) or pick_gender(dict(zip(GENDER_KEYS, gender_p[idx].mean(0))))

    # colours: every image votes (this is how "same design, other colour" adds a colour)
    colors = []
    for i in idx:
        order = np.argsort(color_p[i])[::-1]
        picks = [order[0]] + [c for c in order[1:2] if color_p[i][c] >= SECOND_COLOR_MIN]
        for c in picks:
            if COLORS[c] not in colors:
                colors.append(COLORS[c])

    allowed = BAG_MOTIFS if category == "Bags" else MOTIFS
    mp = {m: float(np.mean([motif_p[i].get(m, 0.0) for i in idx])) for m in allowed}
    best = max(mp, key=mp.get)
    motif = best if mp[best] >= MOTIF_MIN else ""
    stone = keyword(caption, CAPTION_STONE)
    if stone and stone != motif and category != "Bags":
        motif = stone if not motif or motif == "Minimalist" else motif

    if category == "Bags":
        size = {"Mini": "Small", "Tote": "Large", "Backpack": "Large"}.get(motif, DEFAULT_SIZES["Bags"])
        sizes = size
    else:
        sizes = DEFAULT_SIZES[category]
        size = sizes
    gm = GM_TYPICAL[category]

    main_color = colors[0] if colors else ""
    name = " ".join(x for x in [main_color, motif, SINGULAR[category]] if x)
    who = {"Women": "women", "Men": "men", "Kids": "kids", "Unisex": "men and women"}[gender]
    desc = (f"{name} for {who}" +
            (f", also available in {', '.join(colors[1:])}" if len(colors) > 1 else "") +
            ". A stylish piece for everyday wear or special occasions.")

    return {
        "name": name,
        "desc": desc,
        "category": category,
        "image": urls[idx[0]],
        "image_urls": " | ".join(dict.fromkeys(urls[i] for i in idx)),
        "sizes": sizes,
        "colors": ", ".join(colors),
        "gm": gm if gm is not None else "",
        "gender": gender,
        "price": estimate_price(category, gm or 0, size),
        "currency": CURRENCY,
        "post_urls": " | ".join(dict.fromkeys(post_urls[i] for i in idx)),
        "confidence": round(confidence, 2),
    }


GENDER_KEYS = list(GENDER_PROMPTS)


# --------------------------------------------------------------------------- #
# MAIN
# --------------------------------------------------------------------------- #
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default="telegram_images.csv")
    ap.add_argument("--out", default="catalog.csv")
    ap.add_argument("--limit", type=int, default=0, help="only first N posts (test)")
    ap.add_argument("--merge-post", action="store_true", help="one product per Telegram post")
    args = ap.parse_args()

    IMG_DIR.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(args.csv).sort_values(["message_id", "image_number"])
    df["caption"] = df["caption"].fillna("").astype(str)
    if args.limit:
        keep = list(dict.fromkeys(df["message_id"]))[: args.limit]
        df = df[df["message_id"].isin(keep)]
    print(f"{len(df)} images, {df['message_id'].nunique()} posts")

    print("Downloading ...")
    with ThreadPoolExecutor(DOWNLOAD_WORKERS) as ex:
        paths = list(ex.map(download, df["image_url"]))
    df = df.assign(path=paths)
    failed = df["path"].isna().sum()
    if failed:
        print(f"WARNING: {failed} images failed to download and are skipped")
    df = df[df["path"].notna()].reset_index(drop=True)

    imgs = [load_small(p) for p in df["path"]]
    hashes = np.array([phash(im) for im in imgs], dtype=np.uint64)

    clip = Clip()
    lab = Labeler(clip)
    print("Embedding images ...")
    emb = clip.images(imgs)
    gray_emb = clip.images([ImageOps.grayscale(im).convert("RGB") for im in imgs])

    print("Predicting labels ...")
    cat_p = lab.category(emb)
    cats = [CATEGORIES[int(i)] for i in cat_p.argmax(1)]
    gender_p = lab.gender(emb)
    color_p = np.stack([lab.colors(emb[i], cats[i]) for i in range(len(imgs))])
    motif_p = []
    for i in range(len(imgs)):
        names, p = lab.motifs(emb[i], cats[i])
        motif_p.append(dict(zip(names, p)))

    groups, stats = cluster_images(hashes, gray_emb, cats, list(df["message_id"]),
                                   args.merge_post or MERGE_WHOLE_POST)
    print(f"{len(df)} images -> {len(groups)} products  (merged: {dict(stats)})")

    urls, post_urls, captions = list(df["image_url"]), list(df["post_url"]), list(df["caption"])
    rows = [build_row(sorted(g), urls, post_urls, captions, cat_p, gender_p, color_p, motif_p)
            for g in groups]

    out = pd.DataFrame(rows)
    out.to_csv(args.out, index=False, encoding="utf-8-sig")
    Path(args.out).with_suffix(".json").write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\nDone: {len(out)} products -> {args.out}")
    print(out["category"].value_counts().to_string())


if __name__ == "__main__":
    sys.exit(main())


Writing free_catalog.py


## 3. Make sure `telegram_images.csv` is available
On your PC: put it in the same folder as this notebook. On Colab: a file picker opens.

In [13]:
from pathlib import Path
if not Path("../data/telegram_images.csv").exists():
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"Copy telegram_images.csv into: {Path.cwd()}")
print("OK:", Path("../data/telegram_images.csv").resolve())

OK: D:\My_projects\business\data-engineering\notebook\telegram_images.csv


## 4. Test run (10 posts)
The first run downloads the CLIP model (~600 MB). Any error appears right here in the cell output.

In [14]:
%run free_catalog.py --csv telegram_images.csv --out test_catalog.csv --limit 10

85 images, 10 posts


c:\Users\GM\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\GM\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\GM\.cache\huggingface\hub\models--openai--clip-vit-base-patch32. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an admi

CLIP loaded on cpu
Embedding images ...
Predicting labels ...
85 images -> 30 products  (merged: {'same_design': 151, 'same_image': 7})

Done: 30 products -> test_catalog.csv
category
Necklaces      9
Bracelets      9
Earrings       6
Accessories    5
Sets           1


In [15]:
import pandas as pd
pd.set_option("display.max_colwidth", 50, "display.width", 250)
pd.read_csv("./staging/test_catalog.csv")[["name","category","colors","gender","sizes","gm","price","confidence"]]

,name,category,colors,gender,sizes,gm,price,confidence
0,Gold Star Necklace,Necklaces,"Gold, Rose Gold",Women,Adjustable (40-45 cm),12.0,1540,0.80
1,Multicolor Heart Accessory,Accessories,"Multicolor, Pink, Silver, Rose Gold",Women,One size,8.0,1000,0.54
2,Gold Star Necklace,Necklaces,Gold,Women,Adjustable (40-45 cm),12.0,1540,0.82
3,Brown Moon Earrings,Earrings,Brown,Women,One size,5.0,680,0.86
4,Blue Evil Eye Earrings,Earrings,"Blue, Purple, Green, Turquoise, White",Women,One size,5.0,680,0.92
5,Purple Moon Earrings,Earrings,Purple,Women,One size,5.0,680,0.94
6,Purple Earrings,Earrings,Purple,Women,One size,5.0,680,0.99
7,Pink Moon Earrings,Earrings,Pink,Women,One size,5.0,680,0.91
8,Red Earrings,Earrings,Red,Women,One size,5.0,680,0.99
9,Black Bracelet,Bracelets,"Black, White, Silver",Women,Adjustable,15.0,1860,0.93


## 5. Full run (about 5-15 min)
If different items get wrongly merged, raise `DESIGN_SIM` to 0.95 in step 2; if color variants are not merged, lower it to 0.90.

In [16]:
%run free_catalog.py --csv telegram_images.csv --out catalog.csv

1233 images, 222 posts


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 4755.57it/s]


CLIP loaded on cpu
Embedding images ...
Predicting labels ...
1233 images -> 581 products  (merged: {'same_design': 2283, 'same_image': 219})

Done: 581 products -> catalog.csv
category
Necklaces      216
Rings          119
Bracelets       95
Sets            62
Accessories     54
Earrings        29
Bags             6


## 6. Review the result

In [17]:
df = pd.read_csv("../data/catalog.csv")
print(df.shape, "->", Path("../data/catalog.csv").resolve())
display(df.head(20))
display(df[df.confidence < 0.5])   # least certain category predictions - check these first

(581, 13) -> D:\My_projects\business\data-engineering\notebook\catalog.csv


,name,desc,category,image,image_urls,sizes,colors,gm,gender,price,currency,post_urls,confidence
0,Gold Star Necklace,"Gold Star Necklace for women, also available i...",Necklaces,https://cdn4.telesco.pe/file/HOx7oJYEo8cZlc_vz...,https://cdn4.telesco.pe/file/HOx7oJYEo8cZlc_vz...,Adjustable (40-45 cm),"Gold, Rose Gold",12.0,Women,1540,EGP,https://t.me/citysilverandbags925/3,0.80
1,Multicolor Heart Accessory,"Multicolor Heart Accessory for women, also ava...",Accessories,https://cdn4.telesco.pe/file/NO3vIfGMgF5DDyeuv...,https://cdn4.telesco.pe/file/NO3vIfGMgF5DDyeuv...,One size,"Multicolor, Pink, Silver, Rose Gold",8.0,Women,1000,EGP,https://t.me/citysilverandbags925/3,0.54
2,Gold Star Necklace,Gold Star Necklace for women. A stylish piece ...,Necklaces,https://cdn4.telesco.pe/file/p3C-A7Fw-ZG-65FzU...,https://cdn4.telesco.pe/file/p3C-A7Fw-ZG-65FzU...,Adjustable (40-45 cm),Gold,12.0,Women,1540,EGP,https://t.me/citysilverandbags925/3,0.82
3,Brown Moon Earrings,Brown Moon Earrings for women. A stylish piece...,Earrings,https://cdn4.telesco.pe/file/bcZJ483_Hoqm_dfM0...,https://cdn4.telesco.pe/file/bcZJ483_Hoqm_dfM0...,One size,Brown,5.0,Women,680,EGP,https://t.me/citysilverandbags925/12,0.86
4,Blue Evil Eye Earrings,"Blue Evil Eye Earrings for women, also availab...",Earrings,https://cdn4.telesco.pe/file/oPzN4QQOADLU3qeVw...,https://cdn4.telesco.pe/file/oPzN4QQOADLU3qeVw...,One size,"Blue, Purple, Green, Turquoise, White",5.0,Women,680,EGP,https://t.me/citysilverandbags925/12 | https:/...,0.92
5,Purple Moon Earrings,Purple Moon Earrings for women. A stylish piec...,Earrings,https://cdn4.telesco.pe/file/H6MnICvQtrXFEkGar...,https://cdn4.telesco.pe/file/H6MnICvQtrXFEkGar...,One size,Purple,5.0,Women,680,EGP,https://t.me/citysilverandbags925/12,0.94
6,Purple Earrings,Purple Earrings for women. A stylish piece for...,Earrings,https://cdn4.telesco.pe/file/MQrYaLPCd7u3tSBu1...,https://cdn4.telesco.pe/file/MQrYaLPCd7u3tSBu1...,One size,Purple,5.0,Women,680,EGP,https://t.me/citysilverandbags925/22,0.99
7,Pink Moon Earrings,Pink Moon Earrings for women. A stylish piece ...,Earrings,https://cdn4.telesco.pe/file/akAlxrLiRDZyeumcf...,https://cdn4.telesco.pe/file/akAlxrLiRDZyeumcf...,One size,Pink,5.0,Women,680,EGP,https://t.me/citysilverandbags925/22,0.91
8,Red Earrings,Red Earrings for women. A stylish piece for ev...,Earrings,https://cdn4.telesco.pe/file/aUTCP_ZVa3yuX--d0...,https://cdn4.telesco.pe/file/aUTCP_ZVa3yuX--d0...,One size,Red,5.0,Women,680,EGP,https://t.me/citysilverandbags925/22,0.99
9,Black Bracelet,"Black Bracelet for women, also available in Wh...",Bracelets,https://cdn4.telesco.pe/file/qzMdsxHWv4TvMTtYQ...,https://cdn4.telesco.pe/file/qzMdsxHWv4TvMTtYQ...,Adjustable,"Black, White, Silver",15.0,Women,1860,EGP,https://t.me/citysilverandbags925/39 | https:/...,0.91


,name,desc,category,image,image_urls,sizes,colors,gm,gender,price,currency,post_urls,confidence
16,Silver Chain Link Accessory,Silver Chain Link Accessory for women. A styli...,Accessories,https://cdn4.telesco.pe/file/mTixOMwlcSehfJNPr...,https://cdn4.telesco.pe/file/mTixOMwlcSehfJNPr...,One size,Silver,8.0,Women,1000,EGP,https://t.me/citysilverandbags925/69,0.45
18,Silver Chain Link Necklace,Silver Chain Link Necklace for men. A stylish ...,Necklaces,https://cdn4.telesco.pe/file/hCqwzcKFo_swLIjtg...,https://cdn4.telesco.pe/file/hCqwzcKFo_swLIjtg...,Adjustable (40-45 cm),Silver,12.0,Men,1540,EGP,https://t.me/citysilverandbags925/80,0.46
22,White Pearl Necklace,"White Pearl Necklace for men, also available i...",Necklaces,https://cdn4.telesco.pe/file/IhExLtHAR2bvZWTGb...,https://cdn4.telesco.pe/file/IhExLtHAR2bvZWTGb...,Adjustable (40-45 cm),"White, Silver",12.0,Men,1540,EGP,https://t.me/citysilverandbags925/80,0.46
32,Silver Chain Link Accessory,Silver Chain Link Accessory for women. A styli...,Accessories,https://cdn4.telesco.pe/file/CvzVLX2qZPqmpW3tF...,https://cdn4.telesco.pe/file/CvzVLX2qZPqmpW3tF...,One size,Silver,8.0,Women,1000,EGP,https://t.me/citysilverandbags925/116,0.46
41,Silver Ring,Silver Ring for men and women. A stylish piece...,Rings,https://cdn4.telesco.pe/file/eAqBVuGQAg9-S2Q4t...,https://cdn4.telesco.pe/file/eAqBVuGQAg9-S2Q4t...,Adjustable,Silver,5.0,Unisex,680,EGP,https://t.me/citysilverandbags925/192,0.36
...,...,...,...,...,...,...,...,...,...,...,...,...,...
541,White Ring,White Ring for women. A stylish piece for ever...,Rings,https://cdn4.telesco.pe/file/khWWMeNuDcU-1ZgdZ...,https://cdn4.telesco.pe/file/khWWMeNuDcU-1ZgdZ...,Adjustable,White,5.0,Women,680,EGP,https://t.me/citysilverandbags925/1341,0.36
543,Silver Bracelet,Silver Bracelet for men and women. A stylish p...,Bracelets,https://cdn4.telesco.pe/file/UgJ5ZUc_sfQt7XpoX...,https://cdn4.telesco.pe/file/UgJ5ZUc_sfQt7XpoX...,Adjustable,Silver,15.0,Unisex,1860,EGP,https://t.me/citysilverandbags925/1348,0.29
551,Silver Cross Ring,"Silver Cross Ring for women, also available in...",Rings,https://cdn4.telesco.pe/file/AitXuEZO7OShh6MHW...,https://cdn4.telesco.pe/file/AitXuEZO7OShh6MHW...,Adjustable,"Silver, Black",5.0,Women,680,EGP,https://t.me/citysilverandbags925/1375,0.43
553,Gold Star Ring,Gold Star Ring for men and women. A stylish pi...,Rings,https://cdn4.telesco.pe/file/ZdI7t1dyucxIZnOKv...,https://cdn4.telesco.pe/file/ZdI7t1dyucxIZnOKv...,Adjustable,Gold,5.0,Unisex,680,EGP,https://t.me/citysilverandbags925/1375,0.42


In [18]:
try:
    from google.colab import files
    files.download("catalog.csv")
except ImportError:
    print("Saved: catalog.csv and catalog.json in the notebook folder")

Saved: catalog.csv and catalog.json in the notebook folder
